# 29 Forecasts, smoothing and trend

Real-data checks of the groups "Forecasts" and "Smoothing and trend". This notebook holds only simplydata: data read
with `sdata.table`, the words, the numbers read back with simplydata's own words (through `Result`), and the comparison
with `references/29_forecasts_smoothing_trend.csv`; each comparison prints its "Source:" with a link.

Printed references: the trend package vignette (Mann-Kendall test, Sen's slope, seasonal test and slope) and Cobb
(1978), who placed the drop in the Nile's flow at 1898. Elsewhere the help-page example of the R function simplydata
calls or names is run once by `scripts/references/29_forecasts_smoothing_trend.py` on the classic series of those
help pages (HoltWinters, forecast, arima, filter, runmed, sgolayfilt, lowess, smooth.spline, loess, stl, decompose,
acf, Box.test, spdep, urca, tseries, changepoint). Tolerances: relative 1e-10 for closed formulas, 1e-6 for iterative
fits, permutation p-values within 4 Monte Carlo standard errors, printed values to half a unit of their last digit.

The numbers the time charts draw (`autocorrelation_bars`, `cross_correlation_bars`, `lag_dots`, `trend_and_season`
and `forecast`) are read back with `chart.get_numbers_drawn()` (section 7).

In [1]:
import simplydata as sdata

from reference_checks import Comparisons, Result, quiet

cmp = Comparisons("29_forecasts_smoothing_trend")


def series(name):
    return sdata.table(cmp.data_file(name))

## 1. Forecasts

Each method of `table.forecast` against the R function it names, on the help-page series: Holt-Winters (co2 added,
AirPassengers multiplied, Nile without trend or season), "same season last time", "last value", "average", "straight
on", and "arima" (lh AR(1), USAccDeaths airline model, presidents with empty quarters; exact maximum likelihood, as
simplydata documents). Forecasts and interval ends at 95 % (12 steps) and 80 % (6 steps); `nums.forecast` on the plain
numbers; `show="model"`; `check_on_last=12`.

In [2]:
CASES = [("co2", {"repeats_every": 12}, "repeats_every=12"),
         ("AirPassengers", {"repeats_every": 12, "season": "multiplied"}, "repeats_every=12, season='multiplied'"),
         ("Nile", {"trend": False}, "trend=False"),
         ("co2", {"repeats_every": 12, "test_method": "same season last time"}, "repeats_every=12, test_method='same season last time'"),
         ("Nile", {"test_method": "last value"}, "test_method='last value'"),
         ("Nile", {"test_method": "average"}, "test_method='average'"),
         ("Nile", {"test_method": "straight on"}, "test_method='straight on'"),
         ("lh", {"test_method": "arima", "order": (1, 0, 0)}, "test_method='arima', order=(1, 0, 0)"),
         ("USAccDeaths", {"test_method": "arima", "order": (0, 1, 1), "seasonal_order": (0, 1, 1), "repeats_every": 12},
          "test_method='arima', order=(0, 1, 1), seasonal_order=(0, 1, 1), repeats_every=12"),
         ("presidents", {"test_method": "arima", "order": (1, 0, 0)}, "test_method='arima', order=(1, 0, 0)")]
for name, options, label in CASES:
    for level, steps in ((95, 12), (80, 6)):
        f = Result(quiet(series(name).forecast, "value", sorted_by="step", steps_ahead=steps, interval=level, **options))
        check = f"forecast ({label}, steps_ahead={steps}, interval={level})"
        cmp.check_vector(name, check, "forecast", f["forecast"])
        cmp.check_vector(name, check, "low", f[f"forecast {level} % prediction low"])
        cmp.check_vector(name, check, "high", f[f"forecast {level} % prediction high"])
        if level == 95 and name in ("co2", "Nile") and "test_method" not in options:
            numbers = sdata.numbers(Result(series(name))["value"])
            cmp.check_vector(name, check.replace("forecast (", "nums.forecast ("), "forecast",
                             Result(quiet(numbers.forecast, steps_ahead=steps, interval=level, **options))["forecast"])

model = Result(quiet(series("co2").forecast, "value", sorted_by="step", repeats_every=12, show="model"), key="term")
cmp.check_all("co2", "forecast (repeats_every=12, show='model')", {t: model[t, "estimate"] for t in ("alpha", "beta", "gamma", "sum of squared errors")})
held = Result(quiet(series("co2").forecast, "value", sorted_by="step", repeats_every=12, check_on_last=12))
cmp.check_vector("co2", "forecast (repeats_every=12, check_on_last=12)", "forecast of the held-out year", held["forecast"])
cmp.check_vector("co2", "forecast (repeats_every=12, check_on_last=12)", "error (actual minus forecast)", held["error"])

Read "co2.csv" with "," between the columns (guessed). To be sure, give separator=",".


agrees  co2 | forecast (repeats_every=12, steps_ahead=12, interval=95) | forecast [1]: simplydata 365.107894933, R HoltWinters + forecast (stats 4.6.2) 365.107894933; relative difference 1.56e-16 (allowed 1e-06)
      Source: R HoltWinters + forecast (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/HoltWinters.html
agrees  co2 | forecast (repeats_every=12, steps_ahead=12, interval=95) | forecast [2]: simplydata 365.966391338, R HoltWinters + forecast (stats 4.6.2) 365.966391338; relative difference 1.55e-16 (allowed 1e-06)
      Source: R HoltWinters + forecast (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/HoltWinters.html
agrees  co2 | forecast (repeats_every=12, steps_ahead=12, interval=95) | forecast [3]: simplydata 366.734329477, R HoltWinters + forecast (stats 4.6.2) 366.734329477; relative difference 0.00e+00 (allowed 1e-06)
      Source: R HoltWinters + forecast (stats 4.6.2) (run reference) - https://sea

agrees  Nile | forecast (test_method='straight on', steps_ahead=12, interval=95) | forecast [1]: simplydata 736.161616162, R rwf + forecast (forecast 9.0.2) 736.161616162; relative difference 0.00e+00 (allowed 1e-06)
      Source: R rwf + forecast (forecast 9.0.2) (run reference) - https://search.r-project.org/CRAN/refmans/forecast/html/rwf.html
agrees  Nile | forecast (test_method='straight on', steps_ahead=12, interval=95) | forecast [2]: simplydata 732.323232323, R rwf + forecast (forecast 9.0.2) 732.323232323; relative difference 0.00e+00 (allowed 1e-06)
      Source: R rwf + forecast (forecast 9.0.2) (run reference) - https://search.r-project.org/CRAN/refmans/forecast/html/rwf.html
agrees  Nile | forecast (test_method='straight on', steps_ahead=12, interval=95) | forecast [3]: simplydata 728.484848485, R rwf + forecast (forecast 9.0.2) 728.484848485; relative difference 0.00e+00 (allowed 1e-06)
      Source: R rwf + forecast (forecast 9.0.2) (run reference) - https://search.r-proj

## 2. Which method would have done best: `compare_forecasts_for` (R `forecast::accuracy` on the last 12 readings)

In [3]:
for name, options, label in (("co2", {}, "compare_forecasts_for ()"),
                             ("USAccDeaths", {"test_method": ["exponential smoothing", "same season last time", "arima"], "order": (0, 1, 1),
                                              "seasonal_order": (0, 1, 1)},
                              "compare_forecasts_for (test_method=['exponential smoothing', 'same season last time', 'arima'], order=(0, 1, 1), "
                              "seasonal_order=(0, 1, 1))")):
    cf = Result(quiet(series(name).compare_forecasts_for, "value", sorted_by="step", repeats_every=12, **options), key="method")
    for m in cf.index:
        cmp.check_all(name, label, {f"{m}: {k}": cf[m, k] for k in ("MAE", "RMSE", "MAPE", "MASE")})

Read "co2.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  co2 | compare_forecasts_for () | exponential smoothing: MAE: simplydata 0.297610057355, R forecast accuracy (each method made directly, test set as ts) (forecast 9.0.2) 0.297610057355; relative difference 1.87e-16 (allowed 1e-06)
      Source: R forecast accuracy (each method made directly, test set as ts) (forecast 9.0.2) (run reference) - https://search.r-project.org/CRAN/refmans/forecast/html/accuracy.html
agrees  co2 | compare_forecasts_for () | exponential smoothing: RMSE: simplydata 0.373935506138, R forecast accuracy (each method made directly, test set as ts) (forecast 9.0.2) 0.373935506138; relative difference 1.48e-16 (allowed 1e-06)
      Source: R forecast accuracy (each method made directly, test set as ts) (forecast 9.0.2) (run reference) - https://search.r-project.org/CRAN/refmans/forecast/html/accuracy.html
agrees  co2 | compare_forecasts_for () | exponential smoothing: MAPE: 

agrees  co2 | compare_forecasts_for () | last value: MASE: simplydata 1.61634238485, R forecast accuracy (each method made directly, test set as ts) (forecast 9.0.2) 1.61634238485; relative difference 1.37e-16 (allowed 1e-06)
      Source: R forecast accuracy (each method made directly, test set as ts) (forecast 9.0.2) (run reference) - https://search.r-project.org/CRAN/refmans/forecast/html/accuracy.html
Read "USAccDeaths.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  USAccDeaths | compare_forecasts_for (test_method=['exponential smoothing', 'same season last time', 'arima'], order=(0, 1, 1), seasonal_order=(0, 1, 1)) | arima: MAE: simplydata 231.609388592, R forecast accuracy (each method made directly, test set as ts) (forecast 9.0.2) 231.609388592; relative difference 0.00e+00 (allowed 1e-06)
      Source: R forecast accuracy (each method made directly, test set as ts) (forecast 9.0.2) (run reference) - https://search.r-project.org/CRAN/refmans

## 3. Smoothing: every method on the Nile flow

Moving average (window 5; weights [1, 2, 1]; the ends that have no full window are left out on both sides), median
(R `runmed`), Savitzky-Golay (window 11, degree 2), lowess, spline (`smoothness=1000`, R's lambda, and `"auto"`),
loess with its 95 % band. `of_degree=` for Savitzky-Golay, refused before simplydata 3fd3b85 (FINDINGS A23), is compared with SciPy `savgol_filter`.

In [4]:
def smoothed(**options):
    t = series("Nile")
    quiet(t.add_smoothed_column, "s", from_column="value", sorted_by="step", **options)
    return Result(t)


values = Result(series("Nile"))["value"]
cmp.check_vector("Nile", 'add_smoothed_column (test_method="moving average", window=5)', "smoothed",
                 smoothed(test_method="moving average", window=5)["s"][2:-2])
cmp.check_vector("Nile", 'add_smoothed_column (test_method="moving average", weights=[1, 2, 1])', "smoothed",
                 smoothed(test_method="moving average", weights=[1, 2, 1])["s"][1:-1])
cmp.check_vector("Nile", 'nums.smooth (test_method="median", window=5)', "smoothed", list(quiet(sdata.numbers(values).smooth, test_method="median", window=5)))
cmp.check_vector("Nile", 'add_smoothed_column (test_method="lowess")', "smoothed", smoothed(test_method="lowess")["s"])
cmp.check_vector("Nile", 'add_smoothed_column (test_method="spline", smoothness=1000)', "smoothed", smoothed(test_method="spline", smoothness=1000)["s"])
cmp.check_vector("Nile", 'nums.smooth (test_method="spline", smoothness="auto")', "smoothed",
                 list(quiet(sdata.numbers(values).smooth, test_method="spline", smoothness="auto")))
cmp.check_vector("Nile", 'add_smoothed_column (test_method="savitzky-golay", window=11)', "smoothed", smoothed(test_method="savitzky-golay", window=11)["s"])
cmp.check_vector("Nile", 'nums.smooth (test_method="savitzky-golay", window=11)', "smoothed",
                 list(quiet(sdata.numbers(values).smooth, test_method="savitzky-golay", window=11)))
# of_degree= was refused before simplydata 3fd3b85 (FINDINGS A23); now it is compared
cmp.check_vector("Nile", 'add_smoothed_column (test_method="savitzky-golay", window=11, of_degree=2)', "smoothed",
                 smoothed(test_method="savitzky-golay", window=11, of_degree=2)["s"])
lo = smoothed(test_method="loess", interval=95)
for name, column in (("smoothed", "s"), ("95 % low", "s 95 % CI low"), ("95 % high", "s 95 % CI high")):
    cmp.check_vector("Nile", 'add_smoothed_column (test_method="loess", interval=95)', name, lo[column])

Read "Nile.csv" with "," between the columns (guessed). To be sure, give separator=",".
Read "Nile.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  Nile | add_smoothed_column (test_method="moving average", window=5) | smoothed [1]: simplydata 1122.6, R stats filter (sides = 2) (stats 4.6.2) 1122.6; relative difference 0.00e+00 (allowed 1e-10)
      Source: R stats filter (sides = 2) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/filter.html
agrees  Nile | add_smoothed_column (test_method="moving average", window=5) | smoothed [2]: simplydata 1130.6, R stats filter (sides = 2) (stats 4.6.2) 1130.6; relative difference 0.00e+00 (allowed 1e-10)
      Source: R stats filter (sides = 2) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/filter.html
agrees  Nile | add_smoothed_column (test_method="moving average", window=5) | smoothed [3]: simplydata 1061.2, R stats filter (sides = 2) 

agrees  Nile | add_smoothed_column (test_method="savitzky-golay", window=11) | smoothed [1]: simplydata 1096.51748252, R signal sgolayfilt (degree 2) (signal 1.8.1) 1096.51748252; relative difference 1.45e-15 (allowed 1e-10)
      Source: R signal sgolayfilt (degree 2) (signal 1.8.1) (run reference) - https://search.r-project.org/CRAN/refmans/signal/html/sgolayfilt.html
agrees  Nile | add_smoothed_column (test_method="savitzky-golay", window=11) | smoothed [2]: simplydata 1106.86153846, R signal sgolayfilt (degree 2) (signal 1.8.1) 1106.86153846; relative difference 1.03e-15 (allowed 1e-10)
      Source: R signal sgolayfilt (degree 2) (signal 1.8.1) (run reference) - https://search.r-project.org/CRAN/refmans/signal/html/sgolayfilt.html
agrees  Nile | add_smoothed_column (test_method="savitzky-golay", window=11) | smoothed [3]: simplydata 1115.32913753, R signal sgolayfilt (degree 2) (signal 1.8.1) 1115.32913753; relative difference 6.12e-16 (allowed 1e-10)
      Source: R signal sgolay

## 4. Trend and season; removing a trend

STL (R `stl`, the help page decomposes nottem and co2) with `season_window=7`, `robust=True` and `"fixed"`; moving
averages (R `decompose`, added and multiplied; the trend has no value for the first and last 6 months); straight line
(R `lm` with sum-to-zero month effects); `nums.remove_trend()` and with `breaks=[28]`.

In [5]:
for name, options, label in (("nottem", {"season_window": 7}, "season_window=7"), ("co2", {"season_window": 7, "robust": True}, "season_window=7, robust=True"),
                             ("nottem", {"season_window": "fixed"}, "season_window='fixed'")):
    t = series(name)
    quiet(t.add_trend_and_season, "value", sorted_by="step", repeats_every=12, **options)
    cmp.check_vector(name, f"add_trend_and_season ({label})", "trend", Result(t)["value_trend"])
    cmp.check_vector(name, f"add_trend_and_season ({label})", "season", Result(t)["value_season"])
for name, season in (("co2", "added"), ("AirPassengers", "multiplied")):
    t = series(name)
    quiet(t.add_trend_and_season, "value", sorted_by="step", repeats_every=12, test_method="moving averages", season=season)
    cmp.check_vector(name, f'add_trend_and_season (test_method="moving averages", season="{season}")', "trend", Result(t)["value_trend"][6:-6])
    cmp.check_vector(name, f'add_trend_and_season (test_method="moving averages", season="{season}")', "season", Result(t)["value_season"])
t = series("co2")
quiet(t.add_trend_and_season, "value", sorted_by="step", repeats_every=12, test_method="straight line")
cmp.check_vector("co2", 'add_trend_and_season (test_method="straight line")', "trend", Result(t)["value_trend"])
cmp.check_vector("co2", 'add_trend_and_season (test_method="straight line")', "season", Result(t)["value_season"])
cmp.check_vector("Nile", "nums.remove_trend ()", "detrended", list(quiet(sdata.numbers(values).remove_trend)))
cmp.check_vector("Nile", "nums.remove_trend (breaks=[28])", "detrended", list(quiet(sdata.numbers(values).remove_trend, breaks=[28])))

Read "nottem.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  nottem | add_trend_and_season (season_window=7) | trend [1]: simplydata 48.8993811434, R stats stl(y, s.window = 7) (help page series) (stats 4.6.2) 48.8993811434; relative difference 0.00e+00 (allowed 1e-06)
      Source: R stats stl(y, s.window = 7) (help page series) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/stl.html
agrees  nottem | add_trend_and_season (season_window=7) | trend [2]: simplydata 48.9359244133, R stats stl(y, s.window = 7) (help page series) (stats 4.6.2) 48.9359244133; relative difference 1.45e-16 (allowed 1e-06)
      Source: R stats stl(y, s.window = 7) (help page series) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/stl.html
agrees  nottem | add_trend_and_season (season_window=7) | trend [3]: simplydata 48.9724676833, R stats stl(y, s.window = 7) (help page series) (stats 4.6.2) 48.9724

Read "nottem.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  nottem | add_trend_and_season (season_window='fixed') | trend [1]: simplydata 49.6806726486, R stats stl(y, s.window = "periodic") (help page series) (stats 4.6.2) 49.6806726486; relative difference 1.43e-16 (allowed 1e-06)
      Source: R stats stl(y, s.window = "periodic") (help page series) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/stl.html
agrees  nottem | add_trend_and_season (season_window='fixed') | trend [2]: simplydata 49.545520817, R stats stl(y, s.window = "periodic") (help page series) (stats 4.6.2) 49.545520817; relative difference 0.00e+00 (allowed 1e-06)
      Source: R stats stl(y, s.window = "periodic") (help page series) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/stl.html
agrees  nottem | add_trend_and_season (season_window='fixed') | trend [3]: simplydata 49.4103689855, R stats stl(y, s.

agrees  co2 | add_trend_and_season (test_method="straight line") | trend [1]: simplydata 311.553402785, R stats lm(y ~ t + month), sum-to-zero (stats 4.6.2) 311.553402785; relative difference 5.47e-16 (allowed 1e-10)
      Source: R stats lm(y ~ t + month), sum-to-zero (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/lm.html
agrees  co2 | add_trend_and_season (test_method="straight line") | trend [2]: simplydata 311.66261102, R stats lm(y ~ t + month), sum-to-zero (stats 4.6.2) 311.66261102; relative difference 5.47e-16 (allowed 1e-10)
      Source: R stats lm(y ~ t + month), sum-to-zero (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/lm.html
agrees  co2 | add_trend_and_season (test_method="straight line") | trend [3]: simplydata 311.771819255, R stats lm(y ~ t + month), sum-to-zero (stats 4.6.2) 311.771819255; relative difference 5.47e-16 (allowed 1e-10)
      Source: R stats lm(y ~ t + month), sum-to-zero (stats

## 5. Autocorrelation (lh, the example of R `acf`, and LakeHuron); Moran's I on the globe (quakes)

In [6]:
for name in ("lh", "LakeHuron"):
    for show in ("correlation", "partial", "covariance"):
        a = Result(quiet(series(name).find_autocorrelation_of, "value", sorted_by="step", rows_back=10, show=show))
        column = [c for c in a.columns if show in c][0]
        cmp.check_vector(name, f'find_autocorrelation_of (show="{show}", rows_back=10)', show, a[column])
        n = Result(quiet(sdata.numbers(Result(series(name))["value"]).find_autocorrelation, rows_back=10, show=show))
        cmp.check_vector(name, f'nums.find_autocorrelation (show="{show}", rows_back=10)', show, n[column])
        if show == "correlation":
            cmp.check(name, 'find_autocorrelation_of (show="correlation", rows_back=10)', "Ljung-Box statistic", a.cell(1, "Ljung-Box statistic"))
    b = Result(quiet(series(name).find_autocorrelation_of, "value", sorted_by="step", rows_back=10, band="bartlett"))
    cmp.check_vector(name, 'find_autocorrelation_of (band="bartlett", rows_back=10)', "band high", b["band high"])
for seed, permutations in ((1, 999), (9, 499)):
    m = Result(quiet(series("quakes").find_autocorrelation_of, "mag", x="longitude", y="lat", on_globe=True, nearest=8, permutations=permutations,
                     with_seed=seed)).row(1)
    statistic = [k for k in m if "Moran" in k or k == "statistic"][0]
    cmp.check_all("quakes", f"find_autocorrelation_of (on_globe=True, nearest=8, permutations={permutations}, with_seed={seed})",
                  {"Moran's I": m[statistic], "p-value": m["p-value"]})

Read "lh.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  lh | find_autocorrelation_of (show="correlation", rows_back=10) | correlation [1]: simplydata 0.575524475524, R stats acf (stats 4.6.2) 0.575524475524; relative difference 3.86e-16 (allowed 1e-10)
      Source: R stats acf (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/acf.html
agrees  lh | find_autocorrelation_of (show="correlation", rows_back=10) | correlation [2]: simplydata 0.181818181818, R stats acf (stats 4.6.2) 0.181818181818; relative difference 6.11e-16 (allowed 1e-10)
      Source: R stats acf (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/acf.html
agrees  lh | find_autocorrelation_of (show="correlation", rows_back=10) | correlation [3]: simplydata -0.144755244755, R stats acf (stats 4.6.2) -0.144755244755; relative difference 1.92e-16 (allowed 1e-10)
      Source: R stats acf (stats 4.6.2) (run reference) -

Read "lh.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  lh | find_autocorrelation_of (band="bartlett", rows_back=10) | band high [1]: simplydata 0.282896433519, R plot.acf (ci.type = 'ma'): qnorm(0.975) sqrt((1 + 2 sum r^2) / n) (stats 4.6.2) 0.282896433519; relative difference 0.00e+00 (allowed 1e-10)
      Source: R plot.acf (ci.type = 'ma'): qnorm(0.975) sqrt((1 + 2 sum r^2) / n) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/plot.acf.html
agrees  lh | find_autocorrelation_of (band="bartlett", rows_back=10) | band high [2]: simplydata 0.364756183031, R plot.acf (ci.type = 'ma'): qnorm(0.975) sqrt((1 + 2 sum r^2) / n) (stats 4.6.2) 0.364756183031; relative difference 3.04e-16 (allowed 1e-10)
      Source: R plot.acf (ci.type = 'ma'): qnorm(0.975) sqrt((1 + 2 sum r^2) / n) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/plot.acf.html
agrees  lh | find_autocorrelation_of (ba

explained difference – reference (C24)  quakes | find_autocorrelation_of (on_globe=True, nearest=8, permutations=999, with_seed=1) | Moran's I: simplydata 0.12533592731, R spdep moran.mc (knearneigh longlat = TRUE) (spdep 1.4.3) 0.125695579264; relative difference 2.86e-03 (allowed 1e-10)
      why: spdep's knearneigh(longlat = TRUE) picks other neighbours than great-circle distances on a sphere - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#c24-spdeps-knearneighlonglat--true-picks-other-neighbours-than-great-circle-distances-on-a-sphere
      Source: R spdep moran.mc (knearneigh longlat = TRUE) (spdep 1.4.3) (run reference) - https://search.r-project.org/CRAN/refmans/spdep/html/moran.mc.html
agrees  quakes | find_autocorrelation_of (on_globe=True, nearest=8, permutations=999, with_seed=1) | p-value: simplydata 0.001, R spdep moran.mc (same seed), 4 MCSE (spdep 1.4.3) 0.001; difference 0.00e+00 (allowed 4e-03)
      Source: R spdep moran.mc (same seed), 4 MCS

explained difference – reference (C24)  quakes | find_autocorrelation_of (on_globe=True, nearest=8, permutations=499, with_seed=9) | Moran's I: simplydata 0.12533592731, R spdep moran.mc (knearneigh longlat = TRUE) (spdep 1.4.3) 0.125695579264; relative difference 2.86e-03 (allowed 1e-10)
      why: spdep's knearneigh(longlat = TRUE) picks other neighbours than great-circle distances on a sphere - https://github.com/bioforkgithub/simplydata_tests/blob/main/FINDINGS.md#c24-spdeps-knearneighlonglat--true-picks-other-neighbours-than-great-circle-distances-on-a-sphere
      Source: R spdep moran.mc (knearneigh longlat = TRUE) (spdep 1.4.3) (run reference) - https://search.r-project.org/CRAN/refmans/spdep/html/moran.mc.html
agrees  quakes | find_autocorrelation_of (on_globe=True, nearest=8, permutations=499, with_seed=9) | p-value: simplydata 0.002, R spdep moran.mc (same seed), 4 MCSE (spdep 1.4.3) 0.002; difference 0.00e+00 (allowed 8e-03)
      Source: R spdep moran.mc (same seed), 4 MCS

## 6. Steady or wandering; trend; change points

In [7]:
for name in ("Nile", "LakeHuron"):
    for test_method in ("both", "adf", "kpss"):
        s = Result(quiet(series(name).check_steady_over_time, "value", sorted_by="step", test_method=test_method), key="test")
        if "augmented Dickey-Fuller" in s:
            lag = int(s["augmented Dickey-Fuller", "rows back"])
            cmp.check(name, f"check_steady_over_time: ADF statistic with {lag} rows back", "ADF statistic", s["augmented Dickey-Fuller", "statistic"])
        if "KPSS" in s:
            cmp.check_all(name, f'check_steady_over_time (test_method="{test_method}")', {"KPSS statistic": s["KPSS", "statistic"], "KPSS p-value": s["KPSS", "p-value"]})

q = Result(quiet(series("maxau").find_trend_of, "Q", sorted_by="step")).row(1)
cmp.check_all("maxau Q (trend)", "find_trend_of", {"z": q["z"], "p-value": q["p-value"], "S": q["S"], "Kendall's tau": q["Kendall's tau"]})
s = Result(quiet(series("maxau").find_trend_of, "s", sorted_by="step")).row(1)
cmp.check_all("maxau s (trend)", "find_trend_of", {"Sen's slope": s["slope"], "95 % low": s["95 % CI low"], "95 % high": s["95 % CI high"]})
n = Result(quiet(series("nottem").find_trend_of, "value", sorted_by="step", repeats_every=12)).row(1)
cmp.check_all("nottem (trend)", "find_trend_of (repeats_every=12)", {"z": n["z"], "p-value": n["p-value"], "seasonal Sen's slope (per year)": n["slope"]})

nile = series("Nile")
pieces = Result(quiet(nile.find_change_points_in, "value", sorted_by="step"))
cmp.check("Nile", "find_change_points_in", "year in which the first piece ends", Result(nile).cell(int(pieces.cell(1, "to")) + 1, "time"))
for change, penalty in (("average", "mbic"), ("average", "bic"), ("average", "aic"), ("average", "hannan-quinn"), ("average and spread", "mbic")):
    p = Result(quiet(nile.find_change_points_in, "value", sorted_by="step", change=change, penalty=penalty))
    ours = {"number of pieces": len(p)}
    ours.update({f"piece {i} ends at reading": p.cell(i, "to") + 1 for i in range(1, len(p) + 1)})
    cmp.check_all("Nile", f'find_change_points_in (change="{change}", penalty="{penalty}")', ours)

Read "Nile.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  Nile | check_steady_over_time: ADF statistic with 4 rows back | ADF statistic: simplydata -2.78195812232, R urca ur.df (drift, lag 4) (urca 1.3.4) -2.78195812232; relative difference 1.60e-16 (allowed 1e-10)
      Source: R urca ur.df (drift, lag 4) (urca 1.3.4) (run reference) - https://search.r-project.org/CRAN/refmans/urca/html/ur.df.html
agrees  Nile | check_steady_over_time (test_method="both") | KPSS statistic: simplydata 0.965434907753, R tseries kpss.test (Level) (tseries 0.10.63) 0.965434907753; relative difference 0.00e+00 (allowed 1e-10)
      Source: R tseries kpss.test (Level) (tseries 0.10.63) (run reference) - https://search.r-project.org/CRAN/refmans/tseries/html/kpss.test.html
agrees  Nile | check_steady_over_time (test_method="both") | KPSS p-value: simplydata 0.01, R tseries kpss.test (Level) (tseries 0.10.63) 0.01; relative difference 0.00e+00 (allowed 1e-10)
      Source

agrees  Nile | find_change_points_in | year in which the first piece ends: simplydata 1898, Cobb (1978), The problem of the Nile: conditional solution to a changepoint problem, Biometrika 65(2), 243-251 1898; difference 0.00e+00 (allowed 0e+00)
      Source: Cobb (1978), The problem of the Nile: conditional solution to a changepoint problem, Biometrika 65(2), 243-251 (printed) - https://doi.org/10.1093/biomet/65.2.243
agrees  Nile | find_change_points_in (change="average", penalty="mbic") | number of pieces: simplydata 2, R changepoint cpt.mean (PELT) (changepoint 2.3) 2; difference 0.00e+00 (allowed 0e+00)
      Source: R changepoint cpt.mean (PELT) (changepoint 2.3) (run reference) - https://search.r-project.org/CRAN/refmans/changepoint/html/cpt.mean.html
agrees  Nile | find_change_points_in (change="average", penalty="mbic") | piece 1 ends at reading: simplydata 28, R changepoint cpt.mean (PELT) (changepoint 2.3) 28; difference 0.00e+00 (allowed 0e+00)
      Source: R changepoint cp

## 7. The numbers the time charts draw: `chart.get_numbers_drawn()`

Each chart gives back what it drew as a table (panel, layer, kind, x, y, low, high); the layer is picked by its kind
or panel with simplydata words (through `Result`). Compared with R `acf` (the bar heights at rows back 1 to 10), `ccf`
with the lag sign reversed (simplydata's shift s pairs mdeaths at row t with fdeaths at row t + s), the readings one row
back (R `lag.plot`), the trend of R `stl(co2, s.window = 7)` (the second panel) and R `HoltWinters` + `forecast` (the
dashed forecast line and its 95 % band).

In [8]:
def drawn(chart, **where):
    """The numbers a chart drew (chart.get_numbers_drawn()), the rows of one layer picked by their columns."""
    return Result(quiet(chart.get_numbers_drawn), where=where)


lh_series = series("lh")
bars = drawn(quiet(lh_series.start_chart().autocorrelation_bars, "value", sorted_by="step", rows_back=10), kind="dot")
cmp.check_vector("lh", "chart autocorrelation_bars (rows_back=10)", "bar height", bars["y"])
cmp.check_vector("lh", "chart autocorrelation_bars (rows_back=10)", "bar position (rows back)", bars["x"])

deaths = series("deaths")
stems = drawn(quiet(deaths.start_chart().cross_correlation_bars, "mdeaths", "fdeaths", sorted_by="step", rows_back=12), kind="dot")
cmp.check_vector("mdeaths and fdeaths", "chart cross_correlation_bars (rows_back=12)", "stem height", stems["y"])
cmp.check_vector("mdeaths and fdeaths", "chart cross_correlation_bars (rows_back=12)", "stem position (shift)", stems["x"])

dots = drawn(quiet(lh_series.start_chart().lag_dots, "value", sorted_by="step", rows_back=1), kind="dot")
cmp.check_vector("lh", "chart lag_dots (rows_back=1)", "dot across (the reading one row back)", dots["x"])
cmp.check_vector("lh", "chart lag_dots (rows_back=1)", "dot up (the reading)", dots["y"])

co2_series = series("co2")
trend = drawn(quiet(co2_series.start_chart().trend_and_season, x="step", y="value", repeats_every=12), panel=2, kind="line")
cmp.check_vector("co2", "chart trend_and_season (repeats_every=12)", "trend panel line", trend["y"])

chart = co2_series.start_chart()
quiet(chart.draw_lines, x="step", y="value")
quiet(chart.forecast, repeats_every=12)
line = drawn(chart, layer=2, kind="line")
band = drawn(chart, kind="band")
cmp.check_vector("co2", "chart forecast (repeats_every=12)", "forecast line", line["y"])
cmp.check_vector("co2", "chart forecast (repeats_every=12)", "band low", band["low"])
cmp.check_vector("co2", "chart forecast (repeats_every=12)", "band high", band["high"])

Read "lh.csv" with "," between the columns (guessed). To be sure, give separator=",".
agrees  lh | chart autocorrelation_bars (rows_back=10) | bar height [1]: simplydata 0.575524475524, R stats acf (plot = FALSE) (stats 4.6.2) 0.575524475524; relative difference 3.86e-16 (allowed 1e-10)
      Source: R stats acf (plot = FALSE) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/acf.html
agrees  lh | chart autocorrelation_bars (rows_back=10) | bar height [2]: simplydata 0.181818181818, R stats acf (plot = FALSE) (stats 4.6.2) 0.181818181818; relative difference 6.11e-16 (allowed 1e-10)
      Source: R stats acf (plot = FALSE) (stats 4.6.2) (run reference) - https://search.r-project.org/CRAN/refmans/stats/html/acf.html
agrees  lh | chart autocorrelation_bars (rows_back=10) | bar height [3]: simplydata -0.144755244755, R stats acf (plot = FALSE) (stats 4.6.2) -0.144755244755; relative difference 1.92e-16 (allowed 1e-10)
      Source: R stats acf (plot = FA

In [9]:
cmp.finish()

29_forecasts_smoothing_trend: 6913 comparisons - 6840 agrees, 2 explained difference – reference, 71 explained difference – precision limit
of the SKIPs, 0 are waiting for a simplydata word (see the reason column)


,notebook,dataset,method,quantity,simplydata,reference,reference source,difference,tolerance,tolerance type,score,status,reason,result,finding,explanation,link,kind,url,ran on
0,29_forecasts_smoothing_trend,co2,"forecast (repeats_every=12, steps_ahead=12, in...",forecast [1],365.107895,365.107895,R HoltWinters + forecast (stats 4.6.2),-5.684342e-14,0.000001,relative,1.556894e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
1,29_forecasts_smoothing_trend,co2,"forecast (repeats_every=12, steps_ahead=12, in...",forecast [2],365.966391,365.966391,R HoltWinters + forecast (stats 4.6.2),5.684342e-14,0.000001,relative,1.553242e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
2,29_forecasts_smoothing_trend,co2,"forecast (repeats_every=12, steps_ahead=12, in...",forecast [3],366.734329,366.734329,R HoltWinters + forecast (stats 4.6.2),0.000000e+00,0.000001,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
3,29_forecasts_smoothing_trend,co2,"forecast (repeats_every=12, steps_ahead=12, in...",forecast [4],368.136402,368.136402,R HoltWinters + forecast (stats 4.6.2),5.684342e-14,0.000001,relative,1.544086e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
4,29_forecasts_smoothing_trend,co2,"forecast (repeats_every=12, steps_ahead=12, in...",forecast [5],368.667352,368.667352,R HoltWinters + forecast (stats 4.6.2),5.684342e-14,0.000001,relative,1.541862e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6908,29_forecasts_smoothing_trend,co2,chart forecast (repeats_every=12),band high [8],365.420620,365.420620,"R HoltWinters + forecast (h = 12, level = 95) ...",5.684342e-14,0.000001,relative,1.555561e-16,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
6909,29_forecasts_smoothing_trend,co2,chart forecast (repeats_every=12),band high [9],363.565560,363.565560,"R HoltWinters + forecast (h = 12, level = 95) ...",0.000000e+00,0.000001,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
6910,29_forecasts_smoothing_trend,co2,chart forecast (repeats_every=12),band high [10],363.894826,363.894826,"R HoltWinters + forecast (h = 12, level = 95) ...",0.000000e+00,0.000001,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
6911,29_forecasts_smoothing_trend,co2,chart forecast (repeats_every=12),band high [11],365.412109,365.412109,"R HoltWinters + forecast (h = 12, level = 95) ...",0.000000e+00,0.000001,relative,0.000000e+00,PASS,,agrees,,,,run reference,https://search.r-project.org/CRAN/refmans/stat...,"GitHub Actions, Linux 7.0.0-1020-azure, AMD EP..."
